# Project 9 — Introduction to LLMs (OpenAI API)

## What is a Large Language Model?
An **LLM** is a Transformer-based neural network trained on hundreds of billions of words.
It can chat, summarize, translate, generate code, extract structured data, reason, plan…

| Provider | Model family |
|---|---|
| OpenAI | GPT-4o, GPT-4o-mini, GPT-3.5 |
| Anthropic | Claude (Opus, Sonnet, Haiku) |
| Google | Gemini |
| Meta | LLaMA (open-source) |

## What we'll build
1. A simple chat completion
2. Zero-shot sentiment classifier (no training data!)
3. Text summarizer
4. Translator
5. **Structured JSON** spam detector
6. A multi-turn chatbot with memory

## Setup
```bash
pip install openai python-dotenv
```

Get an API key from <https://platform.openai.com/> and set it as an environment variable:

**Windows (PowerShell)**
```powershell
setx OPENAI_API_KEY "sk-..."
```

**macOS / Linux**
```bash
export OPENAI_API_KEY="sk-..."
```

Or create a `.env` file in this folder:
```
OPENAI_API_KEY=sk-...
```

## Step 1 — Imports & client

In [ ]:
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except ImportError:
    pass

from openai import OpenAI

API_KEY = os.environ.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = 'gpt-4o-mini'   # fast + cheap; use 'gpt-4o' for best quality

if not API_KEY:
    print('WARNING: OPENAI_API_KEY not set — calls will fail.')
else:
    print('Client ready.')

## Example 1 — Simple chat completion

Each call has a list of **messages** with three possible roles:
- `system` — instructions for the model's behavior
- `user` — your prompt
- `assistant` — the model's previous replies (used for multi-turn)

**Important parameters:**
- `temperature` — 0 = deterministic, 1 = creative
- `max_tokens` — caps the length of the response

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful NLP tutor.'},
        {'role': 'user',   'content': 'Explain what tokenization is in NLP, in 2 sentences.'},
    ],
    temperature=0.3,
    max_tokens=150,
)
print(response.choices[0].message.content)

## Example 2 — Zero-shot sentiment classifier

**Zero-shot** = no training data, no labelled examples. We just describe the task in the prompt.

In [ ]:
def classify_sentiment(text):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system',
             'content': 'You are a sentiment classifier. Reply ONLY with one word: '
                        'POSITIVE, NEGATIVE, or NEUTRAL.'},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
        max_tokens=5,
    )
    return response.choices[0].message.content.strip()

for r in ['I absolutely love this product, it changed my life!',
          'Terrible experience, the item arrived broken.',
          "It's okay, nothing special."]:
    print(f'  {r}\n   → {classify_sentiment(r)}\n')

## Example 3 — Summarization

In [ ]:
def summarize(text, max_words=50):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system',
             'content': f"Summarize the user's text in at most {max_words} words."},
            {'role': 'user', 'content': text},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

long_text = ('Natural Language Processing (NLP) is a subfield of artificial '
             'intelligence focused on the interaction between computers and human '
             'language. NLP enables computers to read, understand, and generate '
             'human languages. Modern NLP relies heavily on deep learning, '
             'particularly Transformer-based architectures like BERT and GPT.')
print(summarize(long_text, 30))

## Example 4 — Translation

In [ ]:
def translate(text, target_language='French'):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system',
             'content': f"Translate the user's English text into {target_language}. "
                        'Return only the translation.'},
            {'role': 'user', 'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

eng = 'Machine learning is changing the world rapidly.'
print('FR:', translate(eng, 'French'))
print('HI:', translate(eng, 'Hindi'))
print('ES:', translate(eng, 'Spanish'))

## Example 5 — Structured JSON output

We can ask the model to return **strict JSON** so we can parse it into a Python dict and use it in a pipeline.

In [ ]:
def spam_detector_json(email_text):
    prompt = f'''
Analyze the following email and decide if it is spam.
Return a JSON object with these EXACT fields:
   - "is_spam": true or false
   - "confidence": a number between 0 and 1
   - "reason": short explanation

Email: """{email_text}"""
'''
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system',
             'content': 'You reply with valid JSON only — no Markdown.'},
            {'role': 'user', 'content': prompt},
        ],
        response_format={'type': 'json_object'},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

test_email = ('Congratulations! You have WON a free iPhone. '
              'Click http://win-prize.com to claim now!')
result = spam_detector_json(test_email)
print(json.dumps(result, indent=2))

## Example 6 — Multi-turn chatbot with memory

By appending each user prompt **and** the assistant's reply to the messages list, the model remembers the whole conversation.

In [ ]:
def chatbot_loop():
    history = [
        {'role': 'system',
         'content': 'You are a friendly assistant who teaches NLP concepts in simple words.'},
    ]
    print("Type 'quit' to exit.")
    while True:
        user_msg = input('You: ').strip()
        if user_msg.lower() in {'quit', 'exit', ''}:
            print('Bye!'); break
        history.append({'role': 'user', 'content': user_msg})
        response = client.chat.completions.create(
            model=MODEL, messages=history, temperature=0.7)
        reply = response.choices[0].message.content
        history.append({'role': 'assistant', 'content': reply})
        print(f'Bot: {reply}\n')

# Uncomment to chat live:
# chatbot_loop()

## Classical NLP vs LLMs

| | Classical (TF-IDF + LogReg, LSTM) | LLM (GPT, Claude) |
|---|---|---|
| Need labelled data? | yes | **no** (zero-shot works) |
| One model per task? | yes | **no** (one handles many) |
| Manual feature engineering? | often | **no** |
| Cost | very cheap | per-token |
| Latency | milliseconds | seconds |
| Predictable? | yes | sometimes hallucinates |

**When to use which:**
- High-volume, low-cost tasks → classical NLP
- Low data, complex reasoning → LLM
- Production: combine **both** — LLM for the hard cases, classical for the rest

## Next steps
- **Function calling** — let the LLM call your Python functions
- **Retrieval-Augmented Generation (RAG)** — let it answer over your own documents
- **Fine-tuning** — adapt a smaller model to your domain
- Try Anthropic Claude (`claude-opus-4-7`, `claude-sonnet-4-6`) and compare!